In [ ]:
# -*- coding: utf-8 -*-
"""
CI Instrumentation Lineage Auditor (Advanced YAML Detector + GitHub API, no cloning)
------------------------------------------------------------------------------------
- Reads your stratified sample (must include 'html_url', optional 'AT_pred').
- Walks repository commit history that touched .github/workflows up to your clone date.
- For each commit, downloads workflow YAMLs at that ref and applies your advanced detector:
  * device/trigger patterns, normalization, confidence scoring, Gradle input hints, etc.
- Classifies states: ACTIVE / MANUAL_ONLY / COMMENTED_ONLY / DISABLED / NONE / REMOVED / UNKNOWN.
- Summarizes per-repo timeline (first seen, last active, state at clone, deactivation event).

Outputs:
  ci_instru_lineage.csv  in OUT_DIR
"""

import os, re, io, json, time, random
from pathlib import Path
from datetime import datetime, time as dtime, timedelta, timezone
import pandas as pd

# ======================= CONFIG =======================
SAMPLED_CSV_DIR   = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\2_Stratified_Sampling")
SAMPLED_CSV_NAME  = "stratified_sample_moe10.csv"  # If None -> auto-pick CSV containing 'stratified'
OUT_DIR           = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\CI_Lineage_Analysis")

# Tokens env file (tries both casings)
TOKENS_ENV_PATH   = Path(r"C:\GitHub\Android-Mobile-Apps\all_tokens.env")
if not TOKENS_ENV_PATH.exists():
    TOKENS_ENV_PATH = Path(r"C:\GitHub\Android-Mobile-Apps\All_Tokens.env")

# API controls
REQUEST_TIMEOUT      = 30
REQUESTS_MAX_RETRIES = 3
BACKOFF_BASE_SEC     = 2.0
MAX_COMMITS_PER_REPO = 200   # safety cap (commits that touched workflows)
PER_PAGE_COMMITS     = 100   # GitHub max

# Cap history at clone date (match your snapshot)
USE_CUTOFF          = True
CLONE_DATE_LOCAL    = "2025-08-10"
LOCAL_TZ            = "America/Toronto"

# Progress
VERBOSE         = True
PROGRESS_EVERY  = 1

# ======================= BASIC UTILS =======================
def ts(): return datetime.now().strftime("%Y-%m-%d %H:%M:%S")
def note(msg): print(f"[{ts()}] {msg}", flush=True) if VERBOSE else None
def warn(msg): print(f"[{ts()}] [WARN] {msg}", flush=True)
def info(msg): print(f"[{ts()}] [INFO] {msg}", flush=True)

def auto_pick_stratified_csv(folder: Path) -> Path:
    cand = sorted([p for p in folder.glob("*.csv") if "stratified" in p.name.lower()])
    if not cand: cand = sorted(folder.glob("*.csv"))
    if not cand: raise FileNotFoundError(f"No CSV found in {folder}")
    return cand[0]

def owner_repo_from_url(url: str):
    try:
        u = (url or "").strip().strip("/")
        if "github.com/" in u:
            tail = u.split("github.com/", 1)[1].strip("/")
            parts = tail.split("/")
            if len(parts) >= 2: return parts[0].lower(), parts[1].lower()
    except Exception: pass
    return None, None

def full_name_from_html_url(url: str) -> str | None:
    o, r = owner_repo_from_url(url); return f"{o}/{r}" if (o and r) else None

# tz helpers
try:
    from zoneinfo import ZoneInfo
except Exception:
    ZoneInfo = None

def end_of_day_local_to_utc(date_str: str, tz_name: str):
    try:
        d = datetime.strptime(date_str, "%Y-%m-%d").date()
    except Exception:
        return None
    if ZoneInfo is None:
        return datetime(d.year, d.month, d.day, 23, 59, 59, tzinfo=timezone.utc)
    tz = ZoneInfo(tz_name)
    local_dt = datetime(d.year, d.month, d.day, 23, 59, 59, tzinfo=tz)
    return local_dt.astimezone(timezone.utc)

# ======================= TOKEN ROTATION / HTTP =======================
def load_tokens_from_env_file(path: Path) -> list:
    tokens = {}
    if path and path.exists():
        for raw in path.read_text(encoding="utf-8", errors="ignore").splitlines():
            line = raw.strip()
            if not line or line.startswith(("#",";")) or "=" not in line:
                continue
            k, v = line.split("=", 1)
            k, v = k.strip(), v.strip().strip('"').strip("'")
            if k.upper().startswith("GITHUB_TOKEN_"):
                try: idx = int(k.split("_")[-1])
                except Exception: idx = 999
                tokens[idx] = v
    return [tokens[i] for i in sorted(tokens.keys()) if tokens[i]]

class TokenRotator:
    def __init__(self, tokens: list[str]): self.tokens, self.i = (tokens or []), 0
    def current(self): return (self.tokens[self.i] if self.tokens else None)
    def rotate(self):
        if self.tokens: self.i = (self.i + 1) % len(self.tokens)
    def headers(self, allow_auth=True):
        base = {"Accept": "application/vnd.github+json", "User-Agent": "ci-lineage-advanced/1.0"}
        tok = self.current()
        if allow_auth and tok: base["Authorization"] = f"Bearer {tok}"
        return base

def gh_get(url, rot: TokenRotator, allow_auth=True, timeout=REQUEST_TIMEOUT):
    import requests
    tries = 0
    while True:
        tries += 1
        resp = requests.get(url, headers=rot.headers(allow_auth=allow_auth), timeout=timeout)
        diag = {"status": resp.status_code, "rate_remaining": resp.headers.get("X-RateLimit-Remaining"),
                "rate_reset": resp.headers.get("X-RateLimit-Reset"), "message": None}
        try:
            j = resp.json()
            if isinstance(j, dict) and "message" in j: diag["message"] = j["message"]
        except Exception: pass
        resp._diag = diag
        if resp.status_code in (200,201,204): return resp
        if resp.status_code in (403,429):
            warn(f"Rate/403 on {url}. Rotating token. Diag={diag}")
            rot.rotate()
            if tries >= REQUESTS_MAX_RETRIES * max(1, len(rot.tokens)): return resp
            sleep_s = BACKOFF_BASE_SEC * (2 ** (tries-1)) + random.random()
            time.sleep(min(sleep_s, 30)); continue
        if resp.status_code == 401 and allow_auth:
            return gh_get(url, rot, allow_auth=False, timeout=timeout)
        return resp

# ======================= GITHUB API QUERIES =======================
PER_PAGE_COMMITS = 100
def list_commits_for_workflows(rot: TokenRotator, owner: str, repo: str, until_iso_utc: str, page: int):
    url = (f"https://api.github.com/repos/{owner}/{repo}/commits"
           f"?path=.github/workflows&per_page={PER_PAGE_COMMITS}&page={page}"
           f"{('&until='+until_iso_utc) if until_iso_utc else ''}")
    return gh_get(url, rot, allow_auth=True)

def get_commit(rot: TokenRotator, owner: str, repo: str, sha: str):
    return gh_get(f"https://api.github.com/repos/{owner}/{repo}/commits/{sha}", rot, allow_auth=True)

def list_workflows_at_ref(rot: TokenRotator, owner: str, repo: str, ref: str):
    return gh_get(f"https://api.github.com/repos/{owner}/{repo}/contents/.github/workflows?ref={ref}", rot, allow_auth=True)

def fetch_text(url: str, rot: TokenRotator):
    r = gh_get(url, rot, allow_auth=True)
    if r.status_code == 200:
        try: return r.text
        except Exception: return r.content.decode("utf-8", errors="ignore")
    return None

# ======================= YOUR ADVANCED DETECTOR (ADAPTED) =======================
COMMENT_LINE_RE = re.compile(r'(?m)^\s*(#|//|REM\b|::).*?$')

def strip_comments(raw: str) -> str:
    return COMMENT_LINE_RE.sub("", raw or "")

def normalize_block_keys(text: str) -> str:
    text = re.sub(r'(?mi)^\s*(script|run|command)\s*:\s*(?!\|)\s*(.+)$', r'\2', text)
    text = re.sub(r'(?mi)^\s*(script|run|command)\s*:\s*\|?\s*$', '', text)
    return text

def compile_any(patterns, flags=re.I | re.M):
    return [re.compile(p, flags) for p in patterns]

def any_match(patterns, text: str) -> bool:
    return any(p.search(text) for p in patterns)

def unique_preserve(seq):
    seen, out = set(), []
    for x in seq:
        if x not in seen:
            seen.add(x); out.append(x)
    return out

# Gradle normalization
GRADLE_PREFIX = (
    r'^\s*'
    r'(?:\S+=\S+\s+)*'
    r'(?:sudo\s+)?'
    r'(?:(?:bash|sh)\s+-c[l]?\s+[\'"]?)?'
    r'(?:[^#\n;]*?&&\s+)?'
    r'(?:cd\s+\S+\s+&&\s+)?'
    r'(?:\./|\.\\)?gradle(?:w)?(?:\.bat)?'
)
GRADLE_ANYWHERE = r'(?i)[^\n]*\bgradle(?:w)?(?:\.bat)?[^\n]*'
VAR_HINT = re.compile(r'\${{\s*(?:matrix|env|vars|inputs)\.([^}]+)\s*}}', re.I)

NON_TEST_PREFIX = r'(?:assemble|bundle|package|compile|merge|process|generate|install|uninstall|jacoco|lint|publish|sign|upload)'

DEVICE_SOURCES = [
    ("Real_Device", "adb devices",      [r'(?m)^\s*adb\s+devices\b']),
    ("Real_Device", "adb get-state",    [r'(?m)^\s*adb\s+get-state\b']),
    ("Real_Device", "adb get-serialno", [r'(?m)^\s*adb\s+get-serialno\b']),
    ("Real_Device", "adb -s <serial> (physical)", [r'(?m)^\s*adb\s+-s\s+(?!emulator-\d+\b)(?!localhost:\d+\b)(?!127\.0\.0\.1:\d+\b)\S+\b']),
    ("Real_Device", "adb install",      [r'(?m)^\s*adb\s+install(\s+-r)?\b']),
    ("Real_Device", "adb shell",        [r'(?m)^\s*adb\s+shell\b']),
    ("Real_Device", "adb root",         [r'(?m)^\s*adb\s+root\b']),
    ("Real_Device", "adb settings",     [r'(?m)^\s*adb\s+shell\s+settings\b']),
    ("Real_Device", "adb input",        [r'(?m)^\s*adb\s+shell\s+input\b']),
    ("Real_Device", "adb pm grant",     [r'(?m)^\s*adb\s+shell\s+pm\s+grant\b']),

    ("Emulator", "adb -s emulator-serial", [
        r'(?m)^\s*adb\s+-s\s+emulator-\d+\b',
        r'(?m)^\s*adb\s+-s\s+(?:localhost|127\.0\.0\.1):\d+\b',
    ]),
    ("Emulator", "emulator -avd/@", [r'(?m)^\s*\S*emulator\b[^\n]*\s(-avd|@)\S+']),
    ("Emulator", "android-wait-for-emulator", [r'(?m)^\s*(?:\./)?android-wait-for-emulator\b']),
    ("Emulator", "start-emulator.sh", [r'(?m)^\s*start-emulator\.sh\b']),
    ("Emulator", "android create avd", [r'(?m)^\s*\S*android\b[^\n]*\bcreate\s+avd\b']),
    ("Emulator", "circle-android wait-for-boot",[r'(?m)^\s*circle-android\s+wait-for-boot\b']),
    ("Emulator", "reactivecircus runner", [r'uses:\s*reactivecircus/android-emulator-runner']),
    ("Emulator", "sys-img component", [
        r'(?m)^\s*-\s*sys-img-[^\s]*-android-(?:\d+|\$[A-Z_][A-Z0-9_]*)\b',
        r'(?m)^\s*-\s*sys-img-[^\s]*-google_apis-[^\s]*(?:\d+|\$[A-Z_][A-Z0-9_]*)\b'
    ]),
    ("Emulator", "avdmanager", [r'(?m)^\s*\S*avdmanager\b']),
    ("Emulator", "sdkmanager system-images/emulator", [
        r'^\s*\S*sdkmanager\b[^\n"]*"system-images;android-(?:\d+|\$[A-Z_][A-Z0-9_]*)[^"\n]*"',
        r'^\s*\S*sdkmanager\b[^\n]*\bsystem-images;android-(?:\d+|\$[A-Z_][A-Z0-9_]*)\b',
    ]),
    ("Emulator", "api-level", [r'\bapi[-_ ]?level\b\s*:?\s*\d{2}']),
    ("Emulator", "abi/arch",  [r'\b(abi|arch)\b\s*:?\s*(x86|x86_64|arm64|armeabi)']),
    ("Emulator", "target image", [r'\btarget\s*:\s*(google_apis|google_apis_playstore|aosp.*)']),
    ("Emulator", "device name", [r'\b(avd[-_ ]?name|device)\b\s*:\s*pixel']),
    ("GMD", "managedDevices DSL",       [r'\bmanageddevices?\b']),
    ("GMD", "ManagedVirtualDevice DSL", [r'\bmanagedvirtualdevice\b|\bcom\.android\.build\.api\.dsl\.ManagedVirtualDevice\b']),
    ("GMD", "GMD task mentions",        [r'\bmanageddevice\w*androidtest\b']),
    ("GMD", "GHA gradle arguments/tasks", [
        r'(?mi)^\s*arguments\s*:\s*[:\w-]*manageddevice\w*androidtest\b',
        r'(?mi)^\s*tasks?\s*:\s*[:\w-]*manageddevice\w*androidtest\b'
    ]),
    ("Third_Party_Lab", "gcloud firebase", [r'(?mi)^\s*gcloud(\s+beta)?\s+firebase\s+test\s+android\s+run\b']),
    ("Third_Party_Lab", "saucectl",        [r'(?mi)^\s*saucectl(\s+run)?\b']),
    ("Third_Party_Lab", "browserstack/bstack",[r'\b(browserstack|bstack)\b']),
    ("Third_Party_Lab", "appcenter test",  [r'(?mi)^\s*appcenter\s+test\s+run\s+android\b']),
    ("Third_Party_Lab", "maestro cloud",   [r'(?mi)^\s*maestro\s+cloud\b']),
    ("Third_Party_Lab", "test_matrix/firebase.json", [r'\btest_matrix\.json\b|\bfirebase\.json\b']),
]

TRIGGER_SOURCES_PRIMARY = [
    ("Gradle", "connectedAndroidTest",                 [rf'(?m){GRADLE_PREFIX}[^\n]*\bconnectedandroidtest\b']),
    ("Gradle", "connected.*Android.*",                 [rf'(?m){GRADLE_PREFIX}[^\n]*\bconnected[a-z0-9:._-]*android[a-z0-9:._-]*test\b']),
    ("Gradle", "connectedCheck",                       [rf'(?m){GRADLE_PREFIX}\s+(?::[\w-]+:)*connectedcheck\b']),
    ("Gradle", "connectedAndroidTest (abbr)",          [rf'(?m){GRADLE_PREFIX}[^\n]*\b(?:cat|connectedandroidtest)\b']),
    ("Gradle", "deviceCheck",                          [rf'(?mi){GRADLE_PREFIX}\s+(?::[\w-]+:)*(?:devicecheck|alldevicechecks)\b']),
    ("Gradle", "managedDevice AndroidTest",            [rf'(?mi){GRADLE_PREFIX}[^\n]*\b(?!{NON_TEST_PREFIX})(?:manageddevice|device)[\w:-]*androidtest\b']),
    ("Gradle", "variant/device AndroidTest",           [rf'(?mi){GRADLE_PREFIX}[^\n]*\b(?!{NON_TEST_PREFIX})[\w:-]*androidtest\b']),
    ("Gradle", "plain androidTest",                    [rf'(?mi){GRADLE_PREFIX}[^\n]*\b(?::[\w-]+:)*androidtest\b']),
    ("Gradle", "Spoon",                                [rf'(?mi){GRADLE_PREFIX}[^\n]*\bspoon(?:\w*androidtest)?\b']),
    ("Gradle", "Marathon",                             [rf'(?mi){GRADLE_PREFIX}[^\n]*\bmarathon(?:\w*androidtest)?\b']),
    ("ADB", "am instrument",                           [r'(?mi)^[^\n]*\bam\s+instrument\b']),
    ("Third_Party_Lab", "gcloud firebase",             [r'(?mi)^[^\n]*\bgcloud(?:\s+beta)?\s+firebase\s+test\s+android\s+run\b']),
    ("Third_Party_Lab", "flank",                       [r'(?mi)^[^\n]*\bflank\s+android\s+run\b']),
    ("Third_Party_Lab", "saucectl",                    [r'(?mi)^[^\n]*\bsaucectl(?:\s+run)?\b']),
    ("Third_Party_Lab", "appcenter run",               [r'(?mi)^[^\n]*\bappcenter\s+test\s+run\s+android\b']),
    ("Flutter", "flutter drive",                       [r'(?mi)^[^\n]*\bflutter\s+drive\b']),
    ("Flutter", "flutter test (integration_test)",     [r'(?mi)^[^\n]*\bflutter\s+test\b[^\n]*\bintegration_test\b']),
    ("Flutter", "dart test (integration_test)",        [r'(?mi)^[^\n]*\bdart\s+test\b[^\n]*\bintegration_test\b']),
]

TRIGGER_SOURCES_ANYWHERE = [
    ("Gradle", "connected (anywhere)",                 [rf'(?mi){GRADLE_ANYWHERE}\bconnected[a-z0-9:._-]*android[a-z0-9:._-]*test\b']),
    ("Gradle", "connectedAndroidTest (anywhere)",      [rf'(?mi){GRADLE_ANYWHERE}\b(?:cat|connectedandroidtest)\b']),
    ("Gradle", "deviceCheck (anywhere)",               [rf'(?mi){GRADLE_ANYWHERE}\b(?::[\w-]+:)*(?:devicecheck|alldevicechecks)\b']),
    ("Gradle", "managedDevice AndroidTest (anywhere)", [rf'(?mi){GRADLE_ANYWHERE}\b(?!{NON_TEST_PREFIX})(?:manageddevice|device)[\w:-]*androidtest\b']),
    ("Gradle", "variant/device AndroidTest (anywhere)",[rf'(?mi){GRADLE_ANYWHERE}\b(?!{NON_TEST_PREFIX})[\w:-]*androidtest\b']),
    ("Gradle", "Spoon (anywhere)",                     [rf'(?mi){GRADLE_ANYWHERE}\bspoon(?:\w*androidtest)?\b']),
    ("Gradle", "Marathon (anywhere)",                  [rf'(?mi){GRADLE_ANYWHERE}\bmarathon(?:\w*androidtest)?\b']),
]

GHA_GRADLE_INPUTS = compile_any([
    r'(?mi)^\s*arguments\s*:\s*[:\w-]*connected.*android.*test\b',
    r'(?mi)^\s*arguments\s*:\s*(?::[\w-]+:)*connectedcheck\b',
    r'(?mi)^\s*arguments\s*:\s*(?::[\w-]+:)*(?:devicecheck|alldevicechecks)\b',
    r'(?mi)^\s*arguments\s*:\s*[\w:-]*androidtest\b',
    r'(?mi)^\s*arguments\s*:\s*\bcat\b',
    r'(?mi)^\s*tasks?\s*:\s*[:\w-]*connected.*android.*test\b',
    r'(?mi)^\s*tasks?\s*:\s*(?::[\w-]+:)*connectedcheck\b',
    r'(?mi)^\s*tasks?\s*:\s*(?::[\w-]+:)*(?:devicecheck|alldevicechecks)\b',
    r'(?mi)^\s*tasks?\s*:\s*[\w:-]*androidtest\b',
    r'(?mi)^\s*tasks?\s*:\s*\bcat\b',
])

def variable_hints_connected(line: str) -> bool:
    m = VAR_HINT.search(line)
    if not m: return False
    hint = m.group(1).lower()
    return any(k in hint for k in ["connected","androidtest","device","managed","e2e","ui","espresso","instrument"])

def compile_group_patterns(specs):
    return [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in specs]

DEVICE_PATTERNS = compile_group_patterns(DEVICE_SOURCES)
TRIGGER_PATTERNS_PRIMARY = compile_group_patterns(TRIGGER_SOURCES_PRIMARY)
TRIGGER_PATTERNS_ANYWHERE = compile_group_patterns(TRIGGER_SOURCES_ANYWHERE)

STRONG_DEVICE_LABELS = {
    "emulator -avd/@", "android-wait-for-emulator", "start-emulator.sh",
    "circle-android wait-for-boot", "reactivecircus runner",
    "avdmanager", "sdkmanager system-images/emulator","android create avd",
    "managedDevices DSL", "ManagedVirtualDevice DSL", "GMD task mentions", "GHA gradle arguments/tasks",
    "adb get-state", "adb get-serialno", "adb -s <serial> (physical)",
    "gcloud firebase", "saucectl", "browserstack/bstack", "appcenter test", "maestro cloud",
    "test_matrix/firebase.json",
}
WEAK_DEVICE_LABELS = {"api-level", "abi/arch", "target image", "device name", "sys-img component"}

def collect_hits_with_groups(patterns, text: str):
    labels, groups = [], []
    for grp, lbl, pats in patterns:
        if any_match(pats, text):
            labels.append(lbl); groups.append(grp)
    return unique_preserve(labels), unique_preserve(groups)

def filter_weak_device_hints(labels, groups):
    if not (set(labels) & STRONG_DEVICE_LABELS):
        labels = [l for l in labels if l not in WEAK_DEVICE_LABELS]
        if not labels:
            groups = []
    return labels, groups

EMULATOR_STRONG_LABELS = {
    "emulator -avd/@", "android-wait-for-emulator", "start-emulator.sh",
    "circle-android wait-for-boot", "reactivecircus runner", "avdmanager",
    "sdkmanager system-images/emulator","adb -s emulator-serial"
}
REAL_DEVICE_STRONG_LABELS = {"adb get-state", "adb get-serialno", "adb -s <serial> (physical)"}
REAL_DEVICE_GENERIC_ADB = {"adb devices","adb install", "adb shell", "adb root", "adb settings", "adb input", "adb pm grant"}

def reconcile_emulator_vs_real(labels, groups):
    lbls = set(labels)
    if lbls & EMULATOR_STRONG_LABELS:
        lbls -= REAL_DEVICE_GENERIC_ADB
        labels = [l for l in labels if l in lbls]
        if "Real_Device" in groups:
            has_real_after = bool(set(labels) & REAL_DEVICE_STRONG_LABELS)
            if not has_real_after:
                groups = [g for g in groups if g != "Real_Device"]
    return labels, groups

def hard_emulator_priority(device_labels, device_groups):
    if ("Emulator" in device_groups
        and "Real_Device" in device_groups
        and not (set(device_labels) & REAL_DEVICE_STRONG_LABELS)):
        device_groups = [g for g in device_groups if g != "Real_Device"]
        device_labels = [l for l in device_labels if l not in REAL_DEVICE_GENERIC_ADB]
    return device_labels, device_groups

# Disabled & triggers (for state classification, text-level heuristics)
DISABLED_RX = [re.compile(p, re.I) for p in [
    r"\bon\s*:\s*\{\s*\}",          # on: {}
    r"\bif\s*:\s*false\b",          # job-level hard disable (simple)
]]
ACTIVE_TRIGGERS = ["push", "pull_request", "schedule"]

def triggers_status(text: str) -> str:
    if not text: return "UNKNOWN"
    low = text.lower()
    for rx in DISABLED_RX:
        if rx.search(low): return "DISABLED"
    has_active = any(f"\n  {t}:" in low or f"\n{t}:" in low for t in ACTIVE_TRIGGERS)
    has_manual = ("workflow_dispatch:" in low)
    if has_active: return "ACTIVE"
    if has_manual and not has_active: return "MANUAL_ONLY"
    return "UNKNOWN"

# Marker-only check (active vs commented) for COMMENTED_ONLY detection
INSTRU_MARKERS = [
    r"reactivecircus/android-emulator-runner",
    r"\bconnectedAndroidTest\b", r"connected.*android.*test",
    r"emulator\s+-avd", r"\bavdmanager\b", r"\bsdkmanager\b",
    r"gcloud\s+firebase\s+test\s+android\s+run",
    r"\badb\b", r"\bmanageddevices?\b", r"\bmanagedvirtualdevice\b",
]
INSTRU_RX = [re.compile(p, re.I) for p in INSTRU_MARKERS]

def find_markers_active_vs_commented(text: str):
    if not text: return (0,0)
    active = 0; commented = 0
    for line in text.splitlines():
        L = line.rstrip("\n")
        is_comment = bool(re.match(r"\s*#", L))
        for rx in INSTRU_RX:
            if rx.search(L):
                if is_comment: commented += 1
                else: active += 1
                break
    return (active, commented)

def advanced_detect_on_texts(texts: list[str], at_pred: int | None = None):
    """
    Run your advanced detector across concatenated workflow texts.
    Returns:
      dict: {
        signal (bool),
        confidence ('high'|'medium'|'low'|''),
        reason (str),
        device_labels (list),
        trigger_labels (list),
        trigger_via_prefix (bool),
        trigger_via_anywhere (bool),
        gradle_present (bool)
      }
    """
    raw = "\n\n".join([t for t in texts if t])
    # Primary normalized pass
    content = strip_comments(raw)
    content = re.sub(r'(?m)^\s*-\s*', '', content)  # flatten list dashes
    content = normalize_block_keys(content)

    device_labels, device_groups = collect_hits_with_groups(DEVICE_PATTERNS, content.lower())
    trigger_labels, trigger_groups = collect_hits_with_groups(TRIGGER_PATTERNS_PRIMARY, content.lower())
    trigger_via_prefix = bool(trigger_labels)

    # gradle-build-action inputs → treat as triggers
    if any_match(GHA_GRADLE_INPUTS, content):
        trigger_labels = unique_preserve(trigger_labels + ["gha gradle arguments"])
        trigger_groups = unique_preserve(trigger_groups + ["Gradle"])
        trigger_via_prefix = True

    # variable-hinted gradle lines
    if not trigger_via_prefix:
        for line in content.splitlines():
            if re.search(GRADLE_ANYWHERE, line) and re.search(VAR_HINT, line):
                if variable_hints_connected(line):
                    trigger_labels = unique_preserve(trigger_labels + ["gradle via variable hint"])
                    trigger_groups = unique_preserve(trigger_groups + ["Gradle"])
                    trigger_via_prefix = True
                    break

    # Reconcile/clean device hints
    device_labels, device_groups = filter_weak_device_hints(device_labels, device_groups)
    device_labels, device_groups = reconcile_emulator_vs_real(device_labels, device_groups)
    device_labels, device_groups = hard_emulator_priority(device_labels, device_groups)

    # Fallback pass if no prefix triggers
    trigger_via_anywhere = False
    if not trigger_via_prefix:
        fallback = strip_comments(raw)
        fallback = re.sub(r'(?m)^\s*-\s*', '', fallback)
        fallback = re.sub(r'(?mi)^\s*(?:command|run|script)\s*:\s*\|?\s*', '', fallback)
        fallback = re.sub(r'(?m)^\s*sudo\s+', '', fallback)

        if not device_labels:
            fb_device_labels, fb_device_groups = collect_hits_with_groups(DEVICE_PATTERNS, fallback.lower())
            fb_device_labels, fb_device_groups = filter_weak_device_hints(fb_device_labels, fb_device_groups)
            fb_device_labels, fb_device_groups = reconcile_emulator_vs_real(fb_device_labels, fb_device_groups)
            if fb_device_labels or fb_device_groups:
                device_labels  = unique_preserve(device_labels  + fb_device_labels)
                device_groups  = unique_preserve(device_groups  + fb_device_groups)
                device_labels, device_groups = hard_emulator_priority(device_labels, device_groups)

        fb_trigger_labels, fb_trigger_groups = collect_hits_with_groups(TRIGGER_PATTERNS_ANYWHERE, fallback.lower())
        if any_match(GHA_GRADLE_INPUTS, fallback):
            fb_trigger_labels.append("gha gradle arguments")
            fb_trigger_groups.append("Gradle")

        if fb_trigger_labels:
            trigger_labels = unique_preserve(trigger_labels + fb_trigger_labels)
            trigger_groups = unique_preserve(trigger_groups + fb_trigger_groups)
            trigger_via_anywhere = True

    has_device_setup = bool(device_labels)
    has_test_trigger = bool(trigger_labels)
    gradle_present = bool(re.search(GRADLE_ANYWHERE, content))

    # Final signal & confidence (same policy as your local scanner)
    real_device_groups = {"Emulator", "GMD", "Third_Party_Lab", "Real_Device"}
    has_real_device_group = any(g in real_device_groups for g in device_groups)
    signal = bool(has_test_trigger or (has_device_setup and has_real_device_group))

    if signal:
        if has_test_trigger:
            confidence = "high"
            reason = f"test_trigger: {', '.join(trigger_labels)}"
        elif has_device_setup and has_real_device_group:
            confidence = "medium" if gradle_present else "low"
            reason = f"device_setup: {', '.join(device_labels)}" + ("; gradle present" if gradle_present else "")
        # Optional boost by AT_pred if provided
        if at_pred is not None and int(at_pred) == 1:
            if confidence == "low":
                confidence = "medium"; reason += " + boosted (AndroidTest present)"
            elif confidence == "medium":
                confidence = "high";  reason += " + boosted (AndroidTest present)"
    else:
        confidence, reason = "", ""

    return {
        "signal": signal,
        "confidence": confidence,
        "reason": reason,
        "device_labels": device_labels,
        "trigger_labels": trigger_labels,
        "trigger_via_prefix": trigger_via_prefix,
        "trigger_via_anywhere": trigger_via_anywhere,
        "gradle_present": gradle_present,
        "raw": raw,
    }

# State resolution combining advanced detector + text heuristics
def classify_state_from_texts(texts: list[str], at_pred: int | None = None):
    if not texts or len(texts) == 0:
        return ("REMOVED", "", "")
    adv = advanced_detect_on_texts(texts, at_pred=at_pred)
    raw = adv["raw"]

    # Comment-only check
    active_m, commented_m = find_markers_active_vs_commented(raw)

    # Triggers/disabled check (text-level)
    trig_state_votes = [triggers_status(t) for t in texts if t]
    if any(v == "DISABLED" for v in trig_state_votes):
        trig_state = "DISABLED"
    elif any(v == "ACTIVE" for v in trig_state_votes):
        trig_state = "ACTIVE"
    elif any(v == "MANUAL_ONLY" for v in trig_state_votes):
        trig_state = "MANUAL_ONLY"
    else:
        trig_state = "UNKNOWN"

    # Resolve final state:
    # 1) No markers anywhere?
    if not adv["signal"] and active_m == 0 and commented_m == 0:
        return ("NONE", "", "")

    # 2) Only commented markers?
    if active_m == 0 and commented_m > 0:
        return ("COMMENTED_ONLY", adv["confidence"], adv["reason"])

    # 3) Markers present (adv.signal True or raw active markers), use trigger state
    if trig_state == "DISABLED":
        return ("DISABLED", adv["confidence"], adv["reason"])
    if adv["signal"]:
        if trig_state == "ACTIVE":
            return ("ACTIVE", adv["confidence"], adv["reason"])
        if trig_state == "MANUAL_ONLY":
            return ("MANUAL_ONLY", adv["confidence"], adv["reason"])
        # Unknown triggers but clear signal: treat as ACTIVE-ish (conservative)
        return ("ACTIVE", adv["confidence"], adv["reason"])

    # Fallback: markers present but adv didn't unify—treat as UNKNOWN
    return ("UNKNOWN", adv["confidence"], adv["reason"])

# ======================= MAIN =======================
def main():
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    # Tokens
    tokens = load_tokens_from_env_file(TOKENS_ENV_PATH)
    rot = TokenRotator(tokens)
    if tokens: info(f"Loaded {len(tokens)} GitHub token(s) from {TOKENS_ENV_PATH}")
    else: warn("No tokens found; unauthenticated access is rate-limited and may fail on private repos.")

    # Cutoff
    until_utc = None
    if USE_CUTOFF:
        utc_dt = end_of_day_local_to_utc(CLONE_DATE_LOCAL, LOCAL_TZ)
        until_utc = utc_dt.strftime("%Y-%m-%dT%H:%M:%SZ") if utc_dt else None
        info(f"Commit history capped at local {CLONE_DATE_LOCAL} (end of day) → until={until_utc} UTC")

    # Load stratified sample
    samp_path = (SAMPLED_CSV_DIR / SAMPLED_CSV_NAME) if SAMPLED_CSV_NAME else auto_pick_stratified_csv(SAMPLED_CSV_DIR)
    sample = pd.read_csv(samp_path)
    if "html_url" not in sample.columns:
        for alt in ["url","repo_url","html_urls","html"]:
            if alt in sample.columns: sample = sample.rename(columns={alt:"html_url"}); break
    if "html_url" not in sample.columns:
        raise ValueError("Sample CSV must include 'html_url' column.")
    sample["full_name"] = sample["html_url"].apply(full_name_from_html_url)
    if "AT_pred" not in sample.columns:
        sample["AT_pred"] = 0   # optional boost if present
    repo_list = sorted([r for r in sample["full_name"].dropna().unique().tolist() if isinstance(r, str)])
    at_map = dict(zip(sample["full_name"], sample["AT_pred"]))

    info(f"Repos in sample: {len(repo_list)}")

    out_rows = []

    for idx, full in enumerate(repo_list, start=1):
        t0 = time.time()
        owner, repo = full.split("/", 1)
        at_pred = int(at_map.get(full, 0))

        # 1) Collect commits touching workflows up to cutoff
        shas = []
        page = 1
        while True:
            r = list_commits_for_workflows(rot, owner, repo, until_utc or "", page=page)
            if r.status_code != 200: break
            arr = r.json()
            if not isinstance(arr, list) or not arr: break
            for c in arr:
                sha = c.get("sha")
                if sha: shas.append(sha)
                if len(shas) >= MAX_COMMITS_PER_REPO: break
            if len(shas) >= MAX_COMMITS_PER_REPO: break
            page += 1

        # 2) Attach commit dates
        commits_info = []
        for sha in shas:
            rc = get_commit(rot, owner, repo, sha)
            if rc.status_code != 200: continue
            c = rc.json()
            cdate = (c.get("commit", {}) or {}).get("author", {}).get("date")
            try:
                dt = datetime.strptime(cdate, "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=timezone.utc)
            except Exception:
                dt = None
            commits_info.append({"sha": sha, "date": dt})

        commits_info = [c for c in commits_info if c["sha"]]
        commits_info.sort(key=lambda x: x["date"] or datetime.min.replace(tzinfo=timezone.utc))

        # 3) Classify state per commit
        timeline = []
        for c in commits_info:
            ref = c["sha"]
            dl = list_workflows_at_ref(rot, owner, repo, ref)
            if dl.status_code == 404:
                timeline.append({"sha": ref, "date": c["date"], "state": "REMOVED",
                                 "confidence": "", "reason": ""})
                continue
            elif dl.status_code != 200:
                timeline.append({"sha": ref, "date": c["date"], "state": "UNKNOWN",
                                 "confidence": "", "reason": ""})
                continue

            items = dl.json()
            texts = []
            for it in items:
                if it.get("type") == "file" and str(it.get("name","")).lower().endswith((".yml",".yaml")):
                    raw_url = it.get("download_url")
                    if raw_url:
                        txt = fetch_text(raw_url, rot)
                        if txt is not None: texts.append(txt)

            state, conf, reason = classify_state_from_texts(texts, at_pred=at_pred)
            timeline.append({"sha": ref, "date": c["date"], "state": state,
                             "confidence": conf, "reason": reason})

        # 4) Summarize lineage
        def first_idx(pred):
            for i, t in enumerate(timeline):
                if pred(t): return i
            return None
        def last_idx(pred):
            for i in range(len(timeline)-1, -1, -1):
                if pred(timeline[i]): return i
            return None

        is_present   = lambda t: t["state"] in ("ACTIVE","MANUAL_ONLY","COMMENTED_ONLY","DISABLED")
        is_active    = lambda t: t["state"] == "ACTIVE"
        is_deact     = lambda t: t["state"] in ("MANUAL_ONLY","COMMENTED_ONLY","DISABLED","REMOVED","NONE")

        fi = first_idx(is_present)
        la = last_idx(is_active)
        last_any = last_idx(lambda t: t["state"] in ("ACTIVE","MANUAL_ONLY","COMMENTED_ONLY","DISABLED","NONE","REMOVED","UNKNOWN"))

        first_instru_sha   = timeline[fi]["sha"] if fi is not None else ""
        first_instru_date  = timeline[fi]["date"].isoformat() if fi is not None and timeline[fi]["date"] else ""
        last_active_sha    = timeline[la]["sha"] if la is not None else ""
        last_active_date   = timeline[la]["date"].isoformat() if la is not None and timeline[la]["date"] else ""

        state_at_clone = timeline[last_any]["state"] if last_any is not None else "UNKNOWN"
        confidence_at_clone = timeline[last_any]["confidence"] if last_any is not None else ""
        reason_at_clone = timeline[last_any]["reason"] if last_any is not None else ""

        deact_sha = ""; deact_date = ""; deact_type = ""
        if la is not None:
            for j in range(la+1, len(timeline)):
                if is_deact(timeline[j]):
                    deact_sha  = timeline[j]["sha"]
                    deact_date = timeline[j]["date"].isoformat() if timeline[j]["date"] else ""
                    deact_type = timeline[j]["state"]
                    break

        out_rows.append({
            "full_name": full,
            "commits_scanned": len(timeline),
            "first_instru_sha": first_instru_sha,
            "first_instru_date": first_instru_date,
            "last_active_sha": last_active_sha,
            "last_active_date": last_active_date,
            "state_at_clone": state_at_clone,
            "confidence_at_clone": confidence_at_clone,
            "reason_at_clone": reason_at_clone,
            "deactivation_type_after_last_active": deact_type,
            "deactivation_sha": deact_sha,
            "deactivation_date": deact_date,
        })

        if idx % PROGRESS_EVERY == 0:
            dt = time.time() - t0
            note(f"[{idx}/{len(repo_list)}] {full} | commits={len(timeline)} | "
                 f"state_at_clone={state_at_clone} ({confidence_at_clone}) | {dt:.1f}s")

    # Write output
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    out_df = pd.DataFrame(out_rows)
    out_path = OUT_DIR / "ci_instru_lineage.csv"
    out_df.to_csv(out_path, index=False, encoding="utf-8-sig")
    info(f"Saved -> {out_path}")

if __name__ == "__main__":
    main()


[2025-08-26 20:55:51] [INFO] Loaded 6 GitHub token(s) from C:\GitHub\Android-Mobile-Apps\all_tokens.env
[2025-08-26 20:55:51] [INFO] Commit history capped at local 2025-08-10 (end of day) → until=2025-08-11T03:59:59Z UTC
[2025-08-26 20:55:51] [INFO] Repos in sample: 377
[2025-08-26 20:56:10] [1/377] 100mslive/100ms-android | commits=22 | state_at_clone=NONE () | 19.4s
[2025-08-26 20:56:16] [2/377] 10miaomiao/bili-down-out | commits=7 | state_at_clone=NONE () | 5.9s
[2025-08-26 20:56:17] [3/377] 7heaven/shswitchview | commits=0 | state_at_clone=UNKNOWN () | 0.3s
[2025-08-26 20:58:28] [4/377] a-mabe/openhiit | commits=116 | state_at_clone=ACTIVE (high) | 131.2s
[2025-08-26 20:58:30] [5/377] abertschi/ad-free | commits=2 | state_at_clone=NONE () | 1.8s
[2025-08-26 21:00:47] [6/377] ably/ably-flutter | commits=124 | state_at_clone=ACTIVE (high) | 137.6s
[2025-08-26 21:12:01] [7/377] acterglobal/a3 | commits=200 | state_at_clone=ACTIVE (medium) | 673.6s
[2025-08-26 21:12:15] [8/377] adevint